# Paper 26: Auto-Encoding Variational Bayes (VAE)
**Authors**: Diederik P. Kingma & Max Welling (ICLR 2014)

---

## 1. The Core Breakdown: Intractable Posteriors & Discontinuous Latents
Standard autoencoders compress inputs $x$ into deterministic vectors $z = e(x)$. However, because there is no probabilistic density constraint on $z$, the latent space is plagued by **discontinuous voids and holes**. Sampling random vectors $z \sim \mathcal{N}(0, I)$ produces uninterpretable noise.

In a principled latent-variable generative model, data is generated by:
$$p_\theta(x) = \int p_\theta(x|z) p(z) \, dz$$
Because evaluating this high-dimensional marginal likelihood and the true posterior $p_\theta(z|x) = \frac{p_\theta(x|z)p(z)}{p_\theta(x)}$ is analytically intractable for deep neural networks, Kingma & Welling introduced **Amortized Variational Inference** using an encoder network $q_\phi(z|x)$.

## 2. The Evidence Lower Bound (ELBO)
We optimize the lower bound on log-likelihood:
$$\log p_\theta(x) \ge \mathcal{L}_{\text{ELBO}}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{\text{KL}}(q_\phi(z|x) \parallel p(z))$$

Minimizing the negative ELBO balances two opposing physical forces:
1. **Reconstruction Loss**: Decodes high-fidelity details from latent codes $z$.
2. **KL Divergence Regularizer**: Pulls the posterior distribution $q_\phi(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$ towards the isotropic Gaussian prior $\mathcal{N}(0, I)$, ensuring smoothness and continuous interpolation.

## 3. The Reparameterization Trick
To backpropagate through the stochastic bottleneck without noisy score-function estimators:
$$\epsilon \sim \mathcal{N}(0, I), \quad z = \mu(x) + \sigma(x) \odot \epsilon = \mu(x) + \exp\left(\frac{1}{2}\log\sigma^2(x)\right) \odot \epsilon$$

---
## Setup & Autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## Unit Tests Execution
Run the full verification suite against your student template or the reference solution.

In [ ]:
import sys
from pathlib import Path

# Add repository root to pythonpath
repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import papers.26_vae.tests as tests
import solutions.26_vae.model as solution_model

print("Testing Reference Solution:")
tests.test_shapes_and_contracts(solution_model)
tests.test_reparameterization_gradient_flow(solution_model)
tests.test_kl_divergence_math_contract(solution_model)
tests.test_overfitting_gate(solution_model)
print("\nAll tests verified successfully on solution!")

---
## Interactive Experiment: Training VAE on Synthetic Geometry Patterns
We generate synthetic 28x28 patterns (vertical stripes, horizontal stripes, centered blocks, and diagonal ramps) and train a convolutional VAE.

In [ ]:
from solutions.26_vae.model import VAE, vae_loss

# Create synthetic training dataset of 8 geometric patterns
x_train = torch.zeros(8, 1, 28, 28, device=device)
x_train[0, 0, :, 12:16] = 1.0           # Vertical stripe
x_train[1, 0, 12:16, :] = 1.0           # Horizontal stripe
x_train[2, 0, 8:20, 8:20] = 1.0         # Center square
x_train[3, 0, :, :] = 0.5               # Uniform gray
x_train[4, 0, :14, :14] = 1.0           # Top-left quad
x_train[5, 0, 14:, 14:] = 1.0           # Bottom-right quad
x_train[6, 0] = torch.linspace(0, 1, 28).unsqueeze(0).repeat(28, 1)  # Gradient H
x_train[7, 0] = torch.linspace(0, 1, 28).unsqueeze(1).repeat(1, 28)  # Gradient V

# Initialize VAE with latent dimension 16
vae = VAE(in_channels=1, latent_dim=16, img_size=28).to(device)
optimizer = optim.Adam(vae.parameters(), lr=5e-3)

history = {"total": [], "recon": [], "kl": []}

print("Training VAE on geometric dataset...")
for epoch in range(1, 101):
    vae.train()
    optimizer.zero_grad()
    recon_x, mu, logvar = vae(x_train)
    total_loss, recon_loss, kl_loss = vae_loss(recon_x, x_train, mu, logvar, beta=0.05)
    total_loss.backward()
    optimizer.step()

    history["total"].append(total_loss.item())
    history["recon"].append(recon_loss.item())
    history["kl"].append(kl_loss.item())

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Total Loss: {total_loss.item():.3f} | Recon: {recon_loss.item():.3f} | KL: {kl_loss.item():.3f}")

# Plot loss trajectories
plt.figure(figsize=(10, 4))
plt.plot(history["total"], label="Total ELBO Loss", lw=2)
plt.plot(history["recon"], label="Reconstruction Loss (MSE)", lw=2)
plt.plot(history["kl"], label="KL Divergence Loss", lw=2, linestyle="--")
plt.title("VAE Training Dynamics (ELBO Optimization)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

---
## Ground Truth vs. VAE Reconstructions

In [ ]:
vae.eval()
with torch.no_grad():
    recon_x, _, _ = vae(x_train)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(x_train[i, 0].cpu().numpy(), cmap="magma")
    axes[0, i].axis("off")
    axes[0, i].set_title(f"Orig {i}")

    axes[1, i].imshow(recon_x[i, 0].cpu().numpy(), cmap="magma")
    axes[1, i].axis("off")
    axes[1, i].set_title(f"Recon {i}")

plt.suptitle("Ground Truth (Top) vs. VAE Reconstruction (Bottom)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Latent Space Interpolation: Exploring the Continuous Manifold
Because the KL divergence regularizer forces $q_\phi(z|x)$ to cover the prior $\mathcal{N}(0, I)$ without voids, linear paths $z(\alpha) = (1 - \alpha) z_1 + \alpha z_2$ produce continuous, smooth shape transformations!

In [ ]:
vae.eval()
with torch.no_grad():
    # Encode pattern 0 (vertical stripe) and pattern 1 (horizontal stripe)
    mu_0, _ = vae.encode(x_train[0:1])
    mu_1, _ = vae.encode(x_train[1:2])

    steps = 10
    interp_images = vae.interpolate(mu_0.squeeze(0), mu_1.squeeze(0), steps=steps)

fig, axes = plt.subplots(1, steps, figsize=(18, 2.5))
alphas = np.linspace(0.0, 1.0, steps)
for idx, ax in enumerate(axes):
    ax.imshow(interp_images[idx, 0].cpu().numpy(), cmap="viridis")
    ax.axis("off")
    ax.set_title(f"$\alpha={alphas[idx]:.1f}$")

plt.suptitle("Continuous Latent Traversal: Vertical Stripe $\to$ Horizontal Stripe", fontsize=13)
plt.tight_layout()
plt.show()

---
## Unconditional Prior Generation: $z \sim \mathcal{N}(0, I)$
Generate brand new samples by drawing random vectors from the isotropic Gaussian prior.

In [ ]:
vae.eval()
with torch.no_grad():
    samples = vae.sample(num_samples=16, device=device)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i, 0].cpu().numpy(), cmap="cividis")
    ax.axis("off")
    ax.set_title(f"Sample {i+1}")

plt.suptitle("Novel Generations Sampled from Prior $p(z) = \mathcal{N}(0, I)$", fontsize=14)
plt.tight_layout()
plt.show()